# Note that this project takes place in R

![Two data scientists working on a dashboard.](hr-image-small.png)

A common problem when creating models to generate business value from data is that the datasets can be so large that it can take days for the model to generate predictions. Ensuring that your dataset is stored as efficiently as possible is crucial for allowing these models to run on a more reasonable timescale without having to reduce the size of the dataset.

You've been hired by a major online data science training provider called *Training Data Ltd.* to clean up one of their largest customer datasets. This dataset will eventually be used to predict whether their students are looking for a new job or not, information that they will then use to direct them to prospective recruiters.

You've been given access to `customer_train.csv`, which is a subset of their entire customer dataset, so you can create a proof-of-concept of a much more efficient storage solution. The dataset contains anonymized student information, and whether they were looking for a new job or not during training:

| Column       | Description                                  |
|------------- |--------------------------------------------- |
| `student_id`   | A unique ID for each student.                 |
| `city`  | A code for the city the student lives in.  |
| `city_development_index` | A scaled development index for the city.       |
| `gender` | The student's gender.       |
| `relevant_experience` | An indicator of the student's work relevant experience.       |
| `enrolled_university` | The type of university course enrolled in (if any).       |
| `education_level` | The student's education level.       |
| `major_discipline` | The educational discipline of the student.       |
| `experience` | The student's total work experience (in years).       |
| `company_size` | The number of employees at the student's current employer.       |
| `last_new_job` | The number of years between the student's current and previous jobs.       |
| `training_hours` | The number of hours of training completed.       |
| `job_change` | An indicator of whether the student is looking for a new job (`1`) or not (`0`).       |


The Head Data Scientist at Training Data Ltd. has asked you to create a data frame called ds_jobs_clean that stores the data in customer_train.csv much more efficiently. Specifically, they have set the following requirements: see Q1-Q7.

If you call `object.size()` on `ds_jobs` and `ds_jobs_clean` after you've preprocessed it, you should notice a substantial decrease in memory usage.

In [58]:
#Load packages
library(readr)
library(dplyr)
library(forcats)
library(stringr)

In [59]:
#read in the data
orig_data = read_csv("customer_train.csv")
orig_data

Rows: 19158 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (10): city, gender, relevant_experience, enrolled_university, education_...
dbl  (4): student_id, city_development_index, training_hours, job_change

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NA,NA,1,36,1
29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0
11561,city_21,0.624,NA,No relevant experience,Full time course,Graduate,STEM,5,NA,NA,never,83,0
33241,city_115,0.789,NA,No relevant experience,NA,Graduate,Business Degree,<1,NA,Pvt Ltd,never,52,1
666,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0
21651,city_176,0.764,NA,Has relevant experience,Part time course,Graduate,STEM,11,NA,NA,1,24,1
28806,city_160,0.920,Male,Has relevant experience,no_enrollment,High School,NA,5,50-99,Funded Startup,1,24,0
402,city_46,0.762,Male,Has relevant experience,no_enrollment,Graduate,STEM,13,<10,Pvt Ltd,>4,18,1
27107,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,7,50-99,Pvt Ltd,1,46,1


In [60]:
#investigate the data
object.size(orig_data)
glimpse(orig_data)

2167296 bytes

Rows: 19,158
Columns: 14
$ student_id             <dbl> 8949, 29725, 11561, 33241, 666, 21651, 28806, 4…
$ city                   <chr> "city_103", "city_40", "city_21", "city_115", "…
$ city_development_index <dbl> 0.920, 0.776, 0.624, 0.789, 0.767, 0.764, 0.920…
$ gender                 <chr> "Male", "Male", NA, NA, "Male", NA, "Male", "Ma…
$ relevant_experience    <chr> "Has relevant experience", "No relevant experie…
$ enrolled_university    <chr> "no_enrollment", "no_enrollment", "Full time co…
$ education_level        <chr> "Graduate", "Graduate", "Graduate", "Graduate",…
$ major_discipline       <chr> "STEM", "STEM", "STEM", "Business Degree", "STE…
$ experience             <chr> ">20", "15", "5", "<1", ">20", "11", "5", "13",…
$ company_size           <chr> NA, "50-99", NA, NA, "50-99", NA, "50-99", "<10…
$ company_type           <chr> NA, "Pvt Ltd", NA, "Pvt Ltd", "Funded Startup",…
$ last_new_job           <chr> "1", ">4", "never", "never", "4", "1", "1", ">4…
$ training_hour

So, the original dataset has a size of 2,167,296 bytes. It has 19,158 rows.

# Q1
Columns containing integer values must be stored as integers (`<int>`).

In [61]:
#Q1
data <- orig_data

#change data type of 'student_id', 'training_hours', 'job_change' to "int"
data$student_id <- as.integer(data$student_id)
data$training_hours <- as.integer(data$training_hours)
data$job_change <- as.integer(data$job_change)

#could also do 'city' & remove the "city_" prefix
    #data$city <- as.integer(sub("^city_", "", data$city))

glimpse(data)

Rows: 19,158
Columns: 14
$ student_id             <int> 8949, 29725, 11561, 33241, 666, 21651, 28806, 4…
$ city                   <chr> "city_103", "city_40", "city_21", "city_115", "…
$ city_development_index <dbl> 0.920, 0.776, 0.624, 0.789, 0.767, 0.764, 0.920…
$ gender                 <chr> "Male", "Male", NA, NA, "Male", NA, "Male", "Ma…
$ relevant_experience    <chr> "Has relevant experience", "No relevant experie…
$ enrolled_university    <chr> "no_enrollment", "no_enrollment", "Full time co…
$ education_level        <chr> "Graduate", "Graduate", "Graduate", "Graduate",…
$ major_discipline       <chr> "STEM", "STEM", "STEM", "Business Degree", "STE…
$ experience             <chr> ">20", "15", "5", "<1", ">20", "11", "5", "13",…
$ company_size           <chr> NA, "50-99", NA, NA, "50-99", NA, "50-99", "<10…
$ company_type           <chr> NA, "Pvt Ltd", NA, "Pvt Ltd", "Funded Startup",…
$ last_new_job           <chr> "1", ">4", "never", "never", "4", "1", "1", ">4…
$ training_hour

# Q2
Columns containing float values must be stored as doubles (`<dbl>`).

The only column containing floats is 'city_development_index', which is already a "dbl" data type.

# Q3
Columns containing categorical data, both ordinal and nominal, must be stored as factors (`<fct>`).

In [63]:
#Q3
#change 'city', 'gender', 'relevant_experience', 'enrolled_university', 'education_level', 'major_discipline', 'experience',
    #'company_size', 'company_type', 'last_new_job' to a "fct" type
data$city <- as.factor(data$city)
data$gender <- as.factor(data$gender)
data$relevant_experience <- as.factor(data$relevant_experience)
data$enrolled_university <- as.factor(data$enrolled_university)
data$education_level <- as.factor(data$education_level)
data$major_discipline <- as.factor(data$major_discipline)
data$experience <- as.factor(data$experience)
data$company_size <- as.factor(data$company_size)
data$company_type <- as.factor(data$company_type)
data$last_new_job <- as.factor(data$last_new_job)

object.size(data)
glimpse(data)

1176416 bytes

Rows: 19,158
Columns: 14
$ student_id             <int> 8949, 29725, 11561, 33241, 666, 21651, 28806, 4…
$ city                   <fct> city_103, city_40, city_21, city_115, city_162,…
$ city_development_index <dbl> 0.920, 0.776, 0.624, 0.789, 0.767, 0.764, 0.920…
$ gender                 <fct> Male, Male, NA, NA, Male, NA, Male, Male, Male,…
$ relevant_experience    <fct> Has relevant experience, No relevant experience…
$ enrolled_university    <fct> no_enrollment, no_enrollment, Full time course,…
$ education_level        <fct> Graduate, Graduate, Graduate, Graduate, Masters…
$ major_discipline       <fct> STEM, STEM, STEM, Business Degree, STEM, STEM, …
$ experience             <fct> >20, 15, 5, <1, >20, 11, 5, 13, 7, 17, 2, 5, >2…
$ company_size           <fct> NA, 50-99, NA, NA, 50-99, NA, 50-99, <10, 50-99…
$ company_type           <fct> NA, Pvt Ltd, NA, Pvt Ltd, Funded Startup, NA, F…
$ last_new_job           <fct> 1, >4, never, never, 4, 1, 1, >4, 1, >4, never,…
$ training_hour

So far, the size of the dataset has been cut by about 45%.

# Q4
Collapse the `company_size` levels to the following:
- `'Micro'`: <10 employees
- `'Small'`: 10-99 employees
- `'Medium'`: 100-999 employees
- `'Large'`: >1000 employees

In doing so, this variable definitively becomes a nominal category.

In [64]:
#Q4

#first investigate the 'company_size' column
levels(data$company_size)

data <- data %>%
    #collapse old factors into new ones
    mutate(company_size = fct_collapse(company_size, "Micro"=c("<10"), "Small"=c("10-49","50-99"), 
                                       "Medium"=c("100-499","500-999"), "Large"=c("1000-4999","5000-9999","10000+")))
#check new factors
levels(data$company_size)

[1] "<10"       "10-49"     "100-499"   "1000-4999" "10000+"    "50-99"    
[7] "500-999"   "5000-9999"

[1] "Micro"  "Small"  "Medium" "Large"

# Q5
Collapse the `experience` levels to the following:
- `'<5'`: <5 year of experience
- `'5-10'`: 5-10 years of experience
- `'>10'`: >10 years of experience

In doing so, this variable definitively becomes a nominal category.

In [65]:
#Q5

#investigate 'experience'
levels(data$experience)

data <- data %>%
    #collapse old factors into new ones
    mutate(experience = fct_collapse(experience,"<5"=c("<1","1","2","3","4"), "5-10"=c("5","6","7","8","9","10"), 
                                     ">10"=c("11","12","13","14","15","16","17","18","19","20",">20")))
#check new factors
levels(data$experience)

[1] "<1"  ">20" "1"   "10"  "11"  "12"  "13"  "14"  "15"  "16"  "17"  "18" 
[13] "19"  "2"   "20"  "3"   "4"   "5"   "6"   "7"   "8"   "9"

[1] "<5"   ">10"  "5-10"

# Q6
The columns of `ds_jobs_clean` must be in the same order as the original dataset.

With the way the data has been manipulated so far, the columns are organized appropriately & do not need any more modification. The only remaining part is officially declaring the variable `ds_jobs_clean`.

In [66]:
#Q6
object.size(data)

1166888 bytes

Recall that the original dataset had a size of 2,167,296 bytes. The data cleaning from Q1-Q6 has reduced the size of the dataset by about 54%.

# Q7
The data frame should be filtered to only contain students with **10 or more years** of experience at companies with **at least 1000 employees**, as their recruiter base is suited to more experienced professionals at large enterprise companies.

In [67]:
#Q7
#filter the data as specified
ds_jobs_clean <- data %>%
    filter(experience == ">10" & company_size == "Large")
ds_jobs_clean
glimpse(ds_jobs_clean)
object.size(ds_jobs_clean)

student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
<int>,<fct>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<int>,<int>
699,city_103,0.920,NA,Has relevant experience,no_enrollment,Graduate,STEM,>10,Large,Pvt Ltd,>4,123,0
25619,city_61,0.913,Male,Has relevant experience,no_enrollment,Graduate,STEM,>10,Large,Pvt Ltd,3,23,0
22293,city_103,0.920,Male,Has relevant experience,Part time course,Graduate,STEM,>10,Large,Pvt Ltd,>4,141,0
26494,city_16,0.910,Male,Has relevant experience,no_enrollment,Graduate,Business Degree,>10,Large,Pvt Ltd,3,145,0
2547,city_114,0.926,Female,Has relevant experience,Full time course,Masters,STEM,>10,Large,Public Sector,2,14,0
25987,city_103,0.920,Other,Has relevant experience,no_enrollment,Graduate,STEM,>10,Large,Public Sector,4,52,1
1180,city_16,0.910,Male,Has relevant experience,no_enrollment,Graduate,STEM,>10,Large,Pvt Ltd,1,52,0
25349,city_16,0.910,Male,Has relevant experience,no_enrollment,Graduate,STEM,>10,Large,Pvt Ltd,>4,28,0
20576,city_97,0.925,Male,Has relevant experience,no_enrollment,Graduate,STEM,>10,Large,Pvt Ltd,>4,19,0


Rows: 1,956
Columns: 14
$ student_id             <int> 699, 25619, 22293, 26494, 2547, 25987, 1180, 25…
$ city                   <fct> city_103, city_61, city_103, city_16, city_114,…
$ city_development_index <dbl> 0.920, 0.913, 0.920, 0.910, 0.926, 0.920, 0.910…
$ gender                 <fct> NA, Male, Male, Male, Female, Other, Male, Male…
$ relevant_experience    <fct> Has relevant experience, Has relevant experienc…
$ enrolled_university    <fct> no_enrollment, no_enrollment, Part time course,…
$ education_level        <fct> Graduate, Graduate, Graduate, Graduate, Masters…
$ major_discipline       <fct> STEM, STEM, STEM, Business Degree, STEM, STEM, …
$ experience             <fct> >10, >10, >10, >10, >10, >10, >10, >10, >10, >1…
$ company_size           <fct> Large, Large, Large, Large, Large, Large, Large…
$ company_type           <fct> Pvt Ltd, Pvt Ltd, Pvt Ltd, Pvt Ltd, Public Sect…
$ last_new_job           <fct> >4, 3, >4, 3, 2, 4, 1, >4, >4, >4, >4, >4, >4, …
$ training_hours

134768 bytes

The filtered dataset has 1,956 rows, which amounts to 134,768 bytes.